In [2]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


In [3]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [4]:
df = pd.read_csv("Spam_emails.csv")

In [5]:
df.head()

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Category  5572 non-null   object
 1   Message   5572 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB


In [7]:
df.isnull().sum()

,0
Category,0
Message,0


In [8]:
df.dropna(inplace=True)

In [9]:
df.drop_duplicates(inplace=True)

In [10]:
print(df.iloc[:,0].value_counts())

Category
ham     4516
spam     641
Name: count, dtype: int64


In [11]:
ps = PorterStemmer()
def clean_text(text):
    text = text.lower()
    text = re.sub('[^a-zA-Z]', ' ', text)
    words = text.split()
    words = [ps.stem(word) for word in words
             if word not in stopwords.words('english')]
    return " ".join(words)

In [12]:
df["Clean_Text"] = df.iloc[:,1].apply(clean_text)


In [13]:
df.iloc[:,0] = df.iloc[:,0].map({"ham":0, "spam":1})

In [14]:
X = df["Clean_Text"]
y = df.iloc[:,0]

In [15]:
tfidf = TfidfVectorizer(max_features=5000)
X = tfidf.fit_transform(X).toarray()


In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [18]:
model = MultinomialNB()
model.fit(X_train, y_train.astype(int))
y_pred = model.predict(X_test.astype(int))

In [20]:
print("Accuracy :", accuracy_score(y_test.astype(int), y_pred))

Accuracy : 0.8682170542635659


In [22]:
print(confusion_matrix(y_test.astype(int), y_pred))

[[896   0]
 [136   0]]


In [26]:
print(classification_report(y_test.astype(int), y_pred))

              precision    recall  f1-score   support

           0       0.87      1.00      0.93       896
           1       0.00      0.00      0.00       136

    accuracy                           0.87      1032
   macro avg       0.43      0.50      0.46      1032
weighted avg       0.75      0.87      0.81      1032



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [28]:
sample = ["Congratulations! You have won a free iPhone. Click here now."]
sample = [clean_text(text) for text in sample]
sample = tfidf.transform(sample).toarray()
prediction = model.predict(sample)
if prediction[0] == 1:
    print("Spam Email")
else:
    print("Not a Spam Email")

Not a Spam Email
